# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 40 • Text Summarization with Pretrained Encoder–Decoder Models

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson develops abstractive text summarization with encoder–decoder
Transformers. It covers source–target preparation, shifted decoder inputs,
teacher forcing, dynamic padding, greedy decoding, beam-search foundations,
length control, repetition control, ROUGE-style evaluation, factuality,
error analysis, checkpointing, and multilingual considerations.

The notebook contains a complete offline CPU experiment plus optional Hugging Face
workflow cells that remain disabled by default.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish extractive and abstractive summarization;
- prepare source documents and target summaries;
- create shifted decoder inputs and labels;
- train an encoder–decoder Transformer summarizer;
- calculate padding-aware sequence loss;
- generate summaries with greedy decoding;
- explain beam search, length penalty, and repetition control;
- calculate ROUGE-1-, ROUGE-2-, and ROUGE-L-like scores;
- analyze compression ratio, omissions, redundancy, and unsupported content;
- structure Hugging Face summarization workflows;
- evaluate Arabic and multilingual summarization requirements.

## Table of Contents

1. Summarization Tasks
2. Extractive and Abstractive Summarization
3. Encoder–Decoder Formulation
4. Shifted Decoder Targets
5. Teacher Forcing
6. Length Control
7. Factuality
8. Offline Corpus
9. Data Splits
10. Tokenization
11. Vocabularies
12. Dataset and Dynamic Padding
13. Positional Encoding
14. Transformer Summarizer
15. Shape Inspection
16. Padding-Aware Loss
17. Training Utilities
18. Model Training
19. Learning Curves
20. Greedy Decoding
21. Qualitative Evaluation
22. Compression Ratio
23. ROUGE-1-Like Evaluation
24. ROUGE-2-Like Evaluation
25. ROUGE-L-Like Evaluation
26. Test Evaluation
27. Error Analysis
28. Omission
29. Redundancy
30. Unsupported Content
31. Beam Search
32. Length Penalty
33. Repetition Control
34. Checkpointing
35. Optional Hugging Face Setup
36. Optional Pipeline
37. Optional AutoModel Generation
38. Optional Trainer Workflow
39. Long Documents
40. Arabic and Multilingual Considerations
41. Reproducibility
42. Knowledge Check
43. Exercises
44. Summary and Next Lesson

# 1. Summarization Tasks

Summarization compresses a longer source into a shorter representation while
preserving important information.

In [ ]:
import copy
import importlib.util
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

tasks = pd.DataFrame(
    [
        ("News", "article", "headline or abstract"),
        ("Scientific", "paper", "abstract"),
        ("Meeting", "transcript", "decisions and actions"),
        ("Clinical", "note", "problem and plan"),
    ],
    columns=["Domain", "Source", "Summary"],
)
tasks

# 2. Extractive and Abstractive Summarization

Extractive summarization selects source spans. Abstractive summarization generates
new text and can paraphrase the source.

In [ ]:
pd.DataFrame(
    [
        ("Extractive", "copies source spans", "lower hallucination risk"),
        ("Abstractive", "generates new text", "greater flexibility and risk"),
    ],
    columns=["Method", "Mechanism", "Property"],
)

# 3. Encoder–Decoder Formulation

The encoder represents the source document. The decoder generates the summary
autoregressively while attending to encoder memory.

# 4. Shifted Decoder Targets

For target:

```text
<BOS> team completed project <EOS>
```

training uses:

```text
decoder input: <BOS> team completed project
labels:         team completed project <EOS>
```

In [ ]:
sequence = ["<BOS>", "team", "completed", "project", "<EOS>"]
pd.DataFrame(
    {"decoder_input": sequence[:-1], "target_label": sequence[1:]}
)

# 5. Teacher Forcing

During training, the decoder receives gold target prefixes. During inference, it
receives its own generated prefixes.

# 6. Length Control

Common controls include minimum length, maximum length, beam search, length
penalty, no-repeat n-gram constraints, and early stopping.

# 7. Factuality

A summary can be fluent but unsupported by the source. Evaluation should separate
relevance, coverage, coherence, compression, and factual consistency.

In [ ]:
pd.DataFrame(
    [
        ("Relevance", "important content retained"),
        ("Coverage", "key facts represented"),
        ("Coherence", "summary reads logically"),
        ("Compression", "summary is shorter"),
        ("Faithfulness", "claims are source-supported"),
    ],
    columns=["Dimension", "Meaning"],
)

# 8. Offline Corpus

In [ ]:
actors = [
    "research team", "medical staff", "software group",
    "university committee", "city council", "engineering unit",
]
actions = ["completed", "approved", "launched", "reviewed", "updated", "tested"]
objects = [
    "the new project", "the safety plan", "the learning platform",
    "the annual report", "the transport system", "the translation model",
]
locations = [
    "in Cairo", "in Boston", "at the university",
    "at the research center", "during the weekly meeting",
    "during field testing",
]
outcomes = [
    "and reported positive results",
    "and identified two remaining issues",
    "and scheduled the next review",
    "and reduced the processing time",
    "and confirmed the expected performance",
    "and requested additional validation",
]

def build_record(index: int) -> dict:
    actor = actors[index % len(actors)]
    action = actions[(index * 2) % len(actions)]
    obj = objects[(index * 3) % len(objects)]
    location = locations[(index * 5) % len(locations)]
    outcome = outcomes[(index * 7) % len(outcomes)]

    source = (
        f"The {actor} {action} {obj} {location} {outcome}. "
        "The members documented the process and shared the findings "
        "with stakeholders."
    )
    summary = f"{actor} {action} {obj} {location} {outcome}"
    return {"source": source, "summary": summary}

dataset = pd.DataFrame([build_record(i) for i in range(180)])
dataset.head()

# 9. Data Splits

In [ ]:
train_frame, test_frame = train_test_split(
    dataset, test_size=0.20, random_state=42
)
train_frame, validation_frame = train_test_split(
    train_frame, test_size=0.20, random_state=42
)

train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

pd.Series(
    {
        "training": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    }
)

# 10. Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)

def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())

tokenize("The research team completed the project.")

# 11. Vocabularies

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

class Vocabulary:
    def __init__(self, texts, include_bos: bool):
        counts = Counter(
            token
            for text in texts
            for token in tokenize(text)
        )
        special_tokens = [PAD_TOKEN, UNK_TOKEN]
        if include_bos:
            special_tokens.append(BOS_TOKEN)
        special_tokens.append(EOS_TOKEN)

        self.index_to_token = special_tokens + sorted(counts)
        self.token_to_index = {
            token: index
            for index, token in enumerate(self.index_to_token)
        }
        self.pad_id = self.token_to_index[PAD_TOKEN]
        self.unk_id = self.token_to_index[UNK_TOKEN]
        self.eos_id = self.token_to_index[EOS_TOKEN]
        self.bos_id = (
            self.token_to_index[BOS_TOKEN]
            if include_bos
            else None
        )

    def __len__(self):
        return len(self.index_to_token)

    def encode(self, text: str, add_bos: bool = False) -> list[int]:
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        ids.extend(
            self.token_to_index.get(token, self.unk_id)
            for token in tokenize(text)
        )
        ids.append(self.eos_id)
        return ids

    def decode(self, token_ids) -> list[str]:
        tokens = []
        for token_id in token_ids:
            token = self.index_to_token[int(token_id)]
            if token == EOS_TOKEN:
                break
            if token not in {PAD_TOKEN, BOS_TOKEN}:
                tokens.append(token)
        return tokens

source_vocabulary = Vocabulary(
    train_frame["source"], include_bos=False
)
target_vocabulary = Vocabulary(
    train_frame["summary"], include_bos=True
)

print("Source vocabulary:", len(source_vocabulary))
print("Target vocabulary:", len(target_vocabulary))

# 12. Dataset and Dynamic Padding

In [ ]:
class SummarizationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {
            "source_ids": torch.tensor(
                source_vocabulary.encode(row["source"]),
                dtype=torch.long,
            ),
            "target_ids": torch.tensor(
                target_vocabulary.encode(row["summary"], add_bos=True),
                dtype=torch.long,
            ),
            "source_text": row["source"],
            "summary_text": row["summary"],
        }

def collate_batch(batch):
    max_source = max(len(item["source_ids"]) for item in batch)
    max_target = max(len(item["target_ids"]) for item in batch)

    source_ids = torch.full(
        (len(batch), max_source),
        source_vocabulary.pad_id,
        dtype=torch.long,
    )
    target_ids = torch.full(
        (len(batch), max_target),
        target_vocabulary.pad_id,
        dtype=torch.long,
    )

    for row, item in enumerate(batch):
        source_ids[row, :len(item["source_ids"])] = item["source_ids"]
        target_ids[row, :len(item["target_ids"])] = item["target_ids"]

    return {
        "source_ids": source_ids,
        "target_ids": target_ids,
        "source_padding_mask": source_ids == source_vocabulary.pad_id,
        "target_padding_mask": target_ids == target_vocabulary.pad_id,
        "source_texts": [item["source_text"] for item in batch],
        "summary_texts": [item["summary_text"] for item in batch],
    }

train_loader = DataLoader(
    SummarizationDataset(train_frame),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)
validation_loader = DataLoader(
    SummarizationDataset(validation_frame),
    batch_size=16,
    shuffle=False,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    SummarizationDataset(test_frame),
    batch_size=16,
    shuffle=False,
    collate_fn=collate_batch,
)

sample_batch = next(iter(train_loader))
print(sample_batch["source_ids"].shape)
print(sample_batch["target_ids"].shape)

# 13. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, model_dimension: int, maximum_length: int = 256):
        super().__init__()
        encoding = torch.zeros(maximum_length, model_dimension)
        positions = torch.arange(
            maximum_length, dtype=torch.float32
        ).unsqueeze(1)
        rates = torch.exp(
            torch.arange(
                0, model_dimension, 2, dtype=torch.float32
            )
            * (-math.log(10000.0) / model_dimension)
        )
        encoding[:, 0::2] = torch.sin(positions * rates)
        encoding[:, 1::2] = torch.cos(positions * rates)
        self.register_buffer("encoding", encoding.unsqueeze(0))

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return embeddings + self.encoding[:, :embeddings.size(1), :]

# 14. Transformer Summarizer

In [ ]:
class TransformerSummarizer(nn.Module):
    def __init__(
        self,
        source_vocabulary_size: int,
        target_vocabulary_size: int,
        model_dimension: int = 48,
        head_count: int = 4,
        encoder_layers: int = 2,
        decoder_layers: int = 2,
        feed_forward_dimension: int = 96,
        dropout: float = 0.10,
    ):
        super().__init__()
        self.model_dimension = model_dimension
        self.source_embedding = nn.Embedding(
            source_vocabulary_size,
            model_dimension,
            padding_idx=source_vocabulary.pad_id,
        )
        self.target_embedding = nn.Embedding(
            target_vocabulary_size,
            model_dimension,
            padding_idx=target_vocabulary.pad_id,
        )
        self.position = PositionalEncoding(model_dimension)
        self.transformer = nn.Transformer(
            d_model=model_dimension,
            nhead=head_count,
            num_encoder_layers=encoder_layers,
            num_decoder_layers=decoder_layers,
            dim_feedforward=feed_forward_dimension,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.output_layer = nn.Linear(
            model_dimension, target_vocabulary_size
        )

    def causal_mask(self, length: int, device: torch.device):
        return torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=device
            ),
            diagonal=1,
        )

    def encode(self, source_ids, source_padding_mask):
        embeddings = (
            self.source_embedding(source_ids)
            * math.sqrt(self.model_dimension)
        )
        return self.transformer.encoder(
            self.position(embeddings),
            src_key_padding_mask=source_padding_mask,
        )

    def decode(
        self,
        target_input_ids,
        memory,
        target_padding_mask,
        source_padding_mask,
    ):
        embeddings = (
            self.target_embedding(target_input_ids)
            * math.sqrt(self.model_dimension)
        )
        return self.transformer.decoder(
            self.position(embeddings),
            memory,
            tgt_mask=self.causal_mask(
                target_input_ids.size(1),
                target_input_ids.device,
            ),
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask,
        )

    def forward(
        self,
        source_ids,
        target_input_ids,
        source_padding_mask,
        target_padding_mask,
    ):
        memory = self.encode(source_ids, source_padding_mask)
        decoded = self.decode(
            target_input_ids,
            memory,
            target_padding_mask,
            source_padding_mask,
        )
        logits = self.output_layer(decoded)
        return {"logits": logits, "memory": memory, "decoded": decoded}

DEVICE = torch.device("cpu")
torch.manual_seed(42)

model = TransformerSummarizer(
    source_vocabulary_size=len(source_vocabulary),
    target_vocabulary_size=len(target_vocabulary),
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(parameter.numel() for parameter in model.parameters()),
)

# 15. Shape Inspection

In [ ]:
source_ids = sample_batch["source_ids"].to(DEVICE)
target_ids = sample_batch["target_ids"].to(DEVICE)
decoder_inputs = target_ids[:, :-1]

with torch.no_grad():
    output = model(
        source_ids,
        decoder_inputs,
        sample_batch["source_padding_mask"].to(DEVICE),
        decoder_inputs == target_vocabulary.pad_id,
    )

print("Memory:", output["memory"].shape)
print("Logits:", output["logits"].shape)

# 16. Padding-Aware Loss

In [ ]:
loss_function = nn.CrossEntropyLoss(
    ignore_index=target_vocabulary.pad_id
)

def sequence_loss(logits, expected_ids):
    return loss_function(
        logits.reshape(-1, logits.size(-1)),
        expected_ids.reshape(-1),
    )

# 17. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate_loss(model, loader):
    model.eval()
    losses = []

    with torch.no_grad():
        for batch in loader:
            source_ids = batch["source_ids"].to(DEVICE)
            target_ids = batch["target_ids"].to(DEVICE)
            decoder_inputs = target_ids[:, :-1]
            expected_ids = target_ids[:, 1:]

            output = model(
                source_ids,
                decoder_inputs,
                batch["source_padding_mask"].to(DEVICE),
                decoder_inputs == target_vocabulary.pad_id,
            )
            losses.append(
                float(
                    sequence_loss(
                        output["logits"], expected_ids
                    ).item()
                )
            )

    return float(np.mean(losses))

# 18. Model Training

In [ ]:
def train_model(
    model,
    epochs: int = 45,
    learning_rate: float = 0.003,
    patience: int = 9,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = float("inf")
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            source_ids = batch["source_ids"].to(DEVICE)
            target_ids = batch["target_ids"].to(DEVICE)
            decoder_inputs = target_ids[:, :-1]
            expected_ids = target_ids[:, 1:]

            optimizer.zero_grad()
            output = model(
                source_ids,
                decoder_inputs,
                batch["source_padding_mask"].to(DEVICE),
                decoder_inputs == target_vocabulary.pad_id,
            )
            loss = sequence_loss(output["logits"], expected_ids)
            loss.backward()

            gradient_norm = clip_grad_norm_(
                model.parameters(), max_norm=5.0
            )
            optimizer.step()

            training_losses.append(float(loss.item()))
            gradient_norms.append(float(gradient_norm))

        validation_loss = evaluate_loss(
            model, validation_loader
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(np.mean(training_losses)),
                "validation_loss": validation_loss,
                "validation_perplexity": math.exp(
                    min(validation_loss, 20.0)
                ),
                "gradient_norm": float(np.mean(gradient_norms)),
            }
        )

        if validation_loss < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            without_improvement = 0
        else:
            without_improvement += 1

        if without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

set_seed(42)
trained_model, training_history = train_model(model)

print("Epochs completed:", len(training_history))
print(
    "Best validation loss:",
    round(training_history["validation_loss"].min(), 4),
)

# 19. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["training_loss"],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Summarization Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 20. Greedy Decoding

In [ ]:
def greedy_summarize(
    model,
    source_text: str,
    maximum_length: int = 24,
    minimum_length: int = 4,
):
    model.eval()

    source_ids = torch.tensor(
        [source_vocabulary.encode(source_text)],
        dtype=torch.long,
        device=DEVICE,
    )
    source_padding_mask = (
        source_ids == source_vocabulary.pad_id
    )

    with torch.no_grad():
        memory = model.encode(
            source_ids, source_padding_mask
        )
        generated = [target_vocabulary.bos_id]
        probabilities = []

        for step in range(maximum_length):
            target_input = torch.tensor(
                [generated],
                dtype=torch.long,
                device=DEVICE,
            )
            decoded = model.decode(
                target_input,
                memory,
                target_input == target_vocabulary.pad_id,
                source_padding_mask,
            )
            logits = model.output_layer(
                decoded[:, -1, :]
            )
            token_probabilities = torch.softmax(
                logits, dim=1
            )

            if step < minimum_length:
                token_probabilities[
                    :, target_vocabulary.eos_id
                ] = 0.0

            next_id = int(
                token_probabilities.argmax(dim=1).item()
            )
            generated.append(next_id)
            probabilities.append(
                float(token_probabilities[0, next_id].item())
            )

            if next_id == target_vocabulary.eos_id:
                break

    return {
        "summary": " ".join(
            target_vocabulary.decode(generated)
        ),
        "probabilities": probabilities,
    }

greedy_summarize(
    trained_model, test_frame.loc[0, "source"]
)

# 21. Qualitative Evaluation

In [ ]:
rows = []
for row in test_frame.head(10).itertuples(index=False):
    result = greedy_summarize(
        trained_model, row.source
    )
    rows.append(
        {
            "source": row.source,
            "reference": row.summary,
            "prediction": result["summary"],
            "mean_confidence": float(
                np.mean(result["probabilities"])
            ),
        }
    )

pd.DataFrame(rows)

# 22. Compression Ratio

In [ ]:
def compression_ratio(source: str, summary: str) -> float:
    return len(tokenize(summary)) / max(
        len(tokenize(source)), 1
    )

compression_ratio(
    test_frame.loc[0, "source"],
    test_frame.loc[0, "summary"],
)

# 23. ROUGE-1-Like Evaluation

In [ ]:
def ngram_counts(tokens: list[str], order: int) -> Counter:
    return Counter(
        tuple(tokens[index:index + order])
        for index in range(len(tokens) - order + 1)
    )

def rouge_n_like(
    reference: str,
    prediction: str,
    order: int,
) -> float:
    reference_ngrams = ngram_counts(
        tokenize(reference), order
    )
    prediction_ngrams = ngram_counts(
        tokenize(prediction), order
    )
    overlap = sum(
        (reference_ngrams & prediction_ngrams).values()
    )
    return overlap / max(
        sum(reference_ngrams.values()), 1
    )

rouge_n_like(
    "team completed project",
    "team completed the project",
    order=1,
)

# 24. ROUGE-2-Like Evaluation

In [ ]:
rouge_n_like(
    "team completed project",
    "team completed the project",
    order=2,
)

# 25. ROUGE-L-Like Evaluation

In [ ]:
def lcs_length(left: list[str], right: list[str]) -> int:
    table = np.zeros(
        (len(left) + 1, len(right) + 1),
        dtype=int,
    )
    for row in range(1, len(left) + 1):
        for column in range(1, len(right) + 1):
            if left[row - 1] == right[column - 1]:
                table[row, column] = (
                    table[row - 1, column - 1] + 1
                )
            else:
                table[row, column] = max(
                    table[row - 1, column],
                    table[row, column - 1],
                )
    return int(table[len(left), len(right)])

def rouge_l_like(
    reference: str,
    prediction: str,
) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    return lcs_length(
        reference_tokens, prediction_tokens
    ) / max(len(reference_tokens), 1)

# 26. Test Evaluation

In [ ]:
def evaluate_summaries(model, frame):
    rows = []
    for row in frame.itertuples(index=False):
        result = greedy_summarize(
            model, row.source
        )
        prediction = result["summary"]
        rows.append(
            {
                "source": row.source,
                "reference": row.summary,
                "prediction": prediction,
                "rouge_1_like": rouge_n_like(
                    row.summary, prediction, 1
                ),
                "rouge_2_like": rouge_n_like(
                    row.summary, prediction, 2
                ),
                "rouge_l_like": rouge_l_like(
                    row.summary, prediction
                ),
                "reference_compression": compression_ratio(
                    row.source, row.summary
                ),
                "prediction_compression": compression_ratio(
                    row.source, prediction
                ),
            }
        )
    return pd.DataFrame(rows)

test_results = evaluate_summaries(
    trained_model, test_frame
)

test_results[
    ["rouge_1_like", "rouge_2_like", "rouge_l_like"]
].mean()

Formal experiments should use established ROUGE implementations and separate
factuality evaluation.

# 27. Error Analysis

In [ ]:
def categorize_error(source, reference, prediction):
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    source_tokens = set(tokenize(source))

    if prediction_tokens == reference_tokens:
        return "correct"
    if len(prediction_tokens) < 0.65 * len(reference_tokens):
        return "omission"
    if len(prediction_tokens) > 1.35 * len(reference_tokens):
        return "too long"

    unsupported = [
        token
        for token in prediction_tokens
        if token not in source_tokens
        and token not in {"the", "a", "an", "and"}
    ]
    if unsupported:
        return "unsupported content"

    repeated_bigrams = [
        bigram
        for bigram, count
        in ngram_counts(
            prediction_tokens, 2
        ).items()
        if count > 1
    ]
    if repeated_bigrams:
        return "redundancy"

    return "paraphrase or ordering"

test_results["error_type"] = [
    categorize_error(source, reference, prediction)
    for source, reference, prediction
    in zip(
        test_results["source"],
        test_results["reference"],
        test_results["prediction"],
    )
]

test_results["error_type"].value_counts()

# 28. Omission

Omission occurs when important source content is absent from the summary.

# 29. Redundancy

Redundancy repeats words, phrases, or facts unnecessarily.

# 30. Unsupported Content

Unsupported content introduces information not grounded in the source.

# 31. Beam Search

Beam search retains multiple partial hypotheses and ranks them by accumulated
log probability.

In [ ]:
pd.DataFrame(
    [
        ("token_ids", "generated prefix"),
        ("log_probability", "accumulated score"),
        ("finished", "EOS emitted"),
        ("normalized_score", "length-adjusted score"),
    ],
    columns=["Field", "Purpose"],
)

# 32. Length Penalty

In [ ]:
def length_penalized_score(
    log_probability: float,
    length: int,
    alpha: float = 0.8,
) -> float:
    return log_probability / max(length, 1) ** alpha

pd.DataFrame(
    [
        (-2.0, 8, length_penalized_score(-2.0, 8)),
        (-2.5, 12, length_penalized_score(-2.5, 12)),
    ],
    columns=["Log probability", "Length", "Score"],
)

# 33. Repetition Control

In [ ]:
pd.DataFrame(
    [
        ("repetition_penalty", "downweight repeated tokens"),
        ("no_repeat_ngram_size", "block repeated n-grams"),
        ("min_length", "avoid premature EOS"),
        ("coverage", "encourage source coverage"),
    ],
    columns=["Control", "Purpose"],
)

# 34. Checkpointing

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = Path(directory) / "summarizer.pt"
    torch.save(
        {
            "model_state_dict": trained_model.state_dict(),
            "source_vocabulary": source_vocabulary.index_to_token,
            "target_vocabulary": target_vocabulary.index_to_token,
        },
        checkpoint_path,
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    reloaded_model = TransformerSummarizer(
        source_vocabulary_size=len(
            checkpoint["source_vocabulary"]
        ),
        target_vocabulary_size=len(
            checkpoint["target_vocabulary"]
        ),
    ).to(DEVICE)
    reloaded_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    reloaded_summary = greedy_summarize(
        reloaded_model,
        test_frame.loc[0, "source"],
    )["summary"]

print(reloaded_summary)

# 35. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec("transformers")
    is not None
)
DATASETS_AVAILABLE = (
    importlib.util.find_spec("datasets")
    is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True
MODEL_ID = "google-t5/t5-small"

pd.Series(
    {
        "transformers installed": TRANSFORMERS_AVAILABLE,
        "datasets installed": DATASETS_AVAILABLE,
        "run demos": RUN_HUGGING_FACE_DEMOS,
        "local files only": USE_LOCAL_FILES_ONLY,
        "model ID": MODEL_ID,
    }
)

# 36. Optional Pipeline

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import pipeline

    summarizer = pipeline(
        task="summarization",
        model=MODEL_ID,
        tokenizer=MODEL_ID,
        device=-1,
        model_kwargs={
            "local_files_only": USE_LOCAL_FILES_ONLY
        },
    )
    output = summarizer(
        test_frame.loc[0, "source"],
        max_new_tokens=40,
        min_new_tokens=8,
        do_sample=False,
    )
    print(output)
else:
    print("Optional summarization pipeline skipped.")

# 37. Optional AutoModel Generation

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import (
        AutoModelForSeq2SeqLM,
        AutoTokenizer,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        local_files_only=USE_LOCAL_FILES_ONLY,
    )
    hf_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        local_files_only=USE_LOCAL_FILES_ONLY,
    ).to("cpu")

    encoded = tokenizer(
        "summarize: " + test_frame.loc[0, "source"],
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )

    with torch.no_grad():
        generated_ids = hf_model.generate(
            **encoded,
            max_new_tokens=40,
            min_new_tokens=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    print(
        tokenizer.decode(
            generated_ids[0],
            skip_special_tokens=True,
        )
    )
else:
    print("Optional AutoModel generation skipped.")

# 38. Optional Trainer Workflow

In [ ]:
trainer_template = '''
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

arguments = Seq2SeqTrainingArguments(
    output_dir="checkpoints/summarization",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=64,
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
'''

print(trainer_template)

# 39. Long Documents

Long documents may require chunking, hierarchical summarization, long-context
models, or retrieval-assisted summarization.

In [ ]:
pd.DataFrame(
    [
        ("Chunk then summarize", "local summaries then merge"),
        ("Hierarchical", "sentence and document representations"),
        ("Long-context model", "single extended input"),
        ("Retrieval-assisted", "select salient passages first"),
    ],
    columns=["Method", "Mechanism"],
)

# 40. Arabic and Multilingual Considerations

Arabic summarization is affected by rich morphology, attached clitics, optional
tashkeel, MSA/dialect variation, subword fragmentation, and named-entity
preservation.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "أَكْمَلَ الْفَرِيقُ الْبَحْثِيُّ التَّجْرِبَةَ وَأَكَّدَ النَّتَائِجَ.",
            "أَكَّدَ الْفَرِيقُ نَجَاحَ التَّجْرِبَةِ.",
        ),
        (
            "رَاجَعَتِ اللَّجْنَةُ التَّقْرِيرَ وَطَلَبَتْ تَحْقِيقًا إِضَافِيًّا.",
            "طَلَبَتِ اللَّجْنَةُ تَحْقِيقًا إِضَافِيًّا.",
        ),
    ],
    columns=["Fully vocalized source", "Fully vocalized summary"],
)
arabic_examples

For fully vocalized Arabic summarization, tashkeel must remain in source, target,
tokenization, generation, and evaluation whenever it is part of the task
definition.

In [ ]:
pd.DataFrame(
    [
        ("Tashkeel", "preserve consistently"),
        ("Clitics", "inspect segmentation and copying"),
        ("Entities", "verify names and places"),
        ("Variety", "separate MSA and dialect evaluation"),
        ("Faithfulness", "inspect unsupported claims"),
    ],
    columns=["Check", "Action"],
)

# 41. Reproducibility

In [ ]:
pd.Series(
    {
        "training examples": len(train_frame),
        "validation examples": len(validation_frame),
        "test examples": len(test_frame),
        "source vocabulary": len(source_vocabulary),
        "target vocabulary": len(target_vocabulary),
        "device": str(DEVICE),
        "seed": 42,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers installed": TRANSFORMERS_AVAILABLE,
        "optional demos enabled": RUN_HUGGING_FACE_DEMOS,
    },
    name="Lesson 40 experiment",
)

# 42. Knowledge Check

1. How do extractive and abstractive summarization differ?
2. Why are target inputs shifted?
3. What is teacher forcing?
4. Why control summary length?
5. What is unsupported summary content?
6. What does compression ratio measure?
7. What does ROUGE-1-like recall measure?
8. How does ROUGE-L-like evaluation use order?
9. What is omission?
10. What is redundancy?
11. Why can beam search prefer short outputs?
12. What does a length penalty do?
13. Why block repeated n-grams?
14. How can long documents be summarized?
15. Why does tashkeel policy matter?

# 43. Exercises

## Exercise 1 — New Dataset
Replace the synthetic corpus with a public summarization dataset.

## Exercise 2 — Length Control
Compare several minimum and maximum summary lengths.

## Exercise 3 — Beam Search
Implement beam search with length normalization.

## Exercise 4 — Repetition Control
Add no-repeat bigram and trigram constraints.

## Exercise 5 — ROUGE
Use an established ROUGE implementation.

## Exercise 6 — Factuality
Create a manual faithfulness annotation form.

## Exercise 7 — Hugging Face Fine-Tuning
Fine-tune `AutoModelForSeq2SeqLM`.

## Exercise 8 — Long Documents
Implement chunked summarization.

## Exercise 9 — Arabic Summarization
Build a fully vocalized MSA summarization dataset.

## Exercise 10 — Model Card
Document intended use, settings, limitations, and evaluation.

## Challenge Exercises

1. Compare T5, BART, and PEGASUS-style summarizers.
2. Add coverage-aware decoding.
3. Evaluate factual consistency with a second model.
4. Compare multilingual and Arabic-specific checkpoints.
5. Build a retrieval-assisted long-document summarizer.

# 44. Summary and Next Lesson

In this lesson:

- extractive and abstractive summarization were distinguished;
- summarization was formulated as conditional generation;
- shifted decoder inputs and padding-aware loss were implemented;
- a complete CPU-only Transformer summarizer was trained;
- greedy decoding generated summaries;
- compression ratio and ROUGE-style metrics were calculated;
- omission, redundancy, unsupported content, and length errors were analyzed;
- beam search, length penalties, and repetition control were introduced;
- optional Hugging Face pipeline, AutoModel, and Trainer workflows were provided;
- long-document, Arabic, multilingual, and tashkeel considerations were integrated.

## Next Lesson

**Lesson 41: Machine Translation with Pretrained Multilingual Transformers**
introduces multilingual translation, language control, BLEU, chrF, COMET workflow
structure, and translation error analysis.

# References

- Hugging Face Transformers documentation: summarization and sequence-to-sequence
  generation.
- Lewis, M. et al. BART.
- Raffel, C. et al. T5.
- Zhang, J. et al. PEGASUS.
- Lin, C.-Y. ROUGE.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.